# Notebook 1 — Edge-case synthesis pipeline

Thin loop on a **real** photo:

`load → edit (Klein) → annotate → API VLM judge → retry`

**Paradigm (after [Notebook 1.5](01.5_method_comparison.ipynb)):**
- **Generate:** `instruct` = Klein on L4 (small local instruction editor)
- **Judge:** API vision chat (Gemini by default) — text/JSON scores, not pixel edit

NB1.5 compares five edit paths; this notebook runs the **production pair** above.

```python
METHOD_BY_ANOMALY = {
    "road_debris": "instruct",   # Klein — default generator
    "traffic_cone": "instruct",
    "fog": "instruct",
}
```

| `HARDWARE` | Generator (`instruct`) | Judge (default) |
|------------|------------------------|------------------|
| `cpu` | InstructPix2Pix | API (`gemini-3-flash-preview`) |
| `gpu_l4` / `gpu_l4x2` | FLUX.2-klein-4B | API (`gemini-3-flash-preview`) |

Override judge to local Qwen: `judge.backend=qwen_vl` in hardware yaml (offline workshops).


---
## 0. Setup

```bash
# From repo root:
uv sync --dev --group edge-case-image-generation

# API key (Vector proxy judge) — copy once, then paste your vp_… key:
cp implementations/edge_case_image_generation/.env.example \
   implementations/edge_case_image_generation/.env
```

Notebooks load `.env` on startup (no terminal `export` needed).

Select the project kernel, then run:


In [ ]:
import sys
from pathlib import Path


def _find_project_root() -> Path:
    here = Path.cwd().resolve()
    search = [here, *here.parents]
    for base in list(search):
        nested = base / "implementations" / "edge_case_image_generation"
        if nested.is_dir():
            search.append(nested)
    for base in search:
        if (base / "src" / "edgecase_synthesis").is_dir() and (base / "configs").is_dir():
            return base
    raise FileNotFoundError("Could not find edge_case_image_generation root")


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)
from edgecase_synthesis.config import load_env

load_env(PROJECT_ROOT)


---
## 1. Load images + choose methods

**Production methods:** `instruct` (Klein, recommended), `inpaint`, `controlnet_dual`.
NB1.5-only methods (`vlm_generate_local`, `vlm_generate_api`) are for comparison — not used here.

Keys = anomalies to run; values = edit method per anomaly.


In [ ]:
import os

# Avoid flaky HF Xet downloads of large models ("Background writer channel closed").
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

from edgecase_synthesis.compare_methods import METHOD_SPECS, PIPELINE_METHODS
from edgecase_synthesis.config import load_config
from edgecase_synthesis.data import (
    ImageSample,
    get_data_source_info,
    list_sample_images,
    prepare_sample_images,
)
from edgecase_synthesis.pipeline import resolve_method_map
from edgecase_synthesis.viz import show_samples
from PIL import Image

DATASET = "mapillary_vistas"  # or rdd2022 | nordland_hf
HARDWARE = "cpu"  # or "gpu_l4" / "gpu_l4x2"

# Production default: Klein instruct (see NB1.5 for ControlNet / inpaint / Qwen / API edit).
METHOD_BY_ANOMALY = {
    "road_debris": "instruct",
    "traffic_cone": "instruct",
    "fog": "instruct",
}
MAX_RETRIES = 2

cfg = load_config(
    start=PROJECT_ROOT,
    overrides=[f"dataset_name={DATASET}", f"hardware={HARDWARE}"],
)
info = get_data_source_info(cfg)
prepare_sample_images(cfg=cfg)
samples_dir = Path(cfg.paths.samples_dir)

workshop = list(METHOD_BY_ANOMALY) or list(cfg.dataset.workshop_anomalies)
method_map = resolve_method_map(METHOD_BY_ANOMALY, workshop, cfg=cfg)
for aid, method in method_map.items():
    if method not in PIPELINE_METHODS:
        raise ValueError(
            f"Method {method!r} for {aid} is NB1.5-only. Use one of {PIPELINE_METHODS} in NB1."
        )

print(f"Dataset:  {cfg.dataset_name}")
print(f"Hardware: {cfg.hardware.name}  family={cfg.generation.family}")
print(f"Source:   {info.label} ({info.license})")
print(f"Generator (instruct): {cfg.generation.instruct_model_id}")
print(f"Judge:    backend={cfg.judge.backend}  model={cfg.judge.model_id}")
print(f"Annotate: {list(cfg.annotation.classes)}")
print("Method plan:")
for aid, method in method_map.items():
    print(f"  {aid:16s} → {method:16s}  ({METHOD_SPECS[method].title})")

scene_paths = sorted(p for p in list_sample_images(samples_dir) if p.stem.startswith("scene_"))
if not scene_paths:
    raise RuntimeError(
        f"No scene_* images in {samples_dir}. Run scripts/extract_mapillary_toy.py"
    )
sample = ImageSample(
    path=scene_paths[0],
    image=Image.open(scene_paths[0]).convert("RGB"),
    name=scene_paths[0].stem,
)
print(f"Seed image: {sample.name}")
print(f"Samples:  {samples_dir}")
show_samples([sample], ncol=1, figsize=(8, 4));


---
## 2. Edit (Klein instruct)

Depth / segmentation run when a method needs them. Default `instruct` uses Klein on L4.


In [ ]:
from edgecase_synthesis.compare_methods import MethodComparer
from edgecase_synthesis.conditioning import DepthEstimator, Segmenter
from edgecase_synthesis.pipeline import synthesize_one
from edgecase_synthesis.viz import save_generation_artifact, show_generation_result

depth_model = DepthEstimator.from_config(cfg)
segmenter = Segmenter.from_config(cfg)
comparer = MethodComparer.from_config(cfg)
print("depth:", depth_model.model_id)
print("seg:  ", segmenter.model_name)
print("edit: ", comparer.family, "on", comparer.device)

depth = depth_model.predict(sample.image)
seg = segmenter.predict(sample.image)

output_dir = Path(cfg.paths.outputs_dir)
results: dict[str, object] = {}

for anomaly_id, method in method_map.items():
    print("=" * 60)
    print(f"{anomaly_id}  via  {method}")
    syn = synthesize_one(
        sample.image,
        anomaly_id=anomaly_id,
        method=method,
        cfg=cfg,
        comparer=comparer,
        depth=depth,
        segmentation=seg,
        project_root=PROJECT_ROOT,
    )
    results[anomaly_id] = syn
    show_generation_result(sample, syn.generated)
    print(save_generation_artifact(sample, syn.generated, output_dir / "nb1" / anomaly_id))


---
## 3. Annotate

YOLO-World open-vocab **boxes** (same path for every edit method). Summaries go to the VLM judge; no edit-mask seeding / SAM.


In [ ]:
from edgecase_synthesis.annotation import OpenVocabAnnotator
from edgecase_synthesis.config import load_anomaly
from edgecase_synthesis.viz import save_annotation_artifact, show_annotation_result


def annotate_anomaly(syn, anomaly_id: str, annotator: OpenVocabAnnotator):
    """Tight per-anomaly queries; optional lower conf for diffuse effects (fog)."""
    anomaly_cfg = load_anomaly(dataset, anomaly_id, start=PROJECT_ROOT)
    anomaly_classes = list(anomaly_cfg.get("annotation_classes", []))
    classes = list(dict.fromkeys([*(anomaly_classes or base_classes)]))
    conf = anomaly_cfg.get("annotation_conf")
    conf = float(conf) if conf is not None else None
    print(f"  queries[{anomaly_id}]: {classes}  conf={conf or cfg.annotation.conf}")
    return annotator.annotate(syn.generated.image, classes=classes, conf=conf)


annotator = OpenVocabAnnotator.from_config(cfg)
base_classes = list(cfg.annotation.classes)
annotations = {}
dataset = str(cfg.dataset_name)
print("detector:", cfg.annotation.detector_model, "conf=", cfg.annotation.conf)
print("dataset vocab:", base_classes)

for anomaly_id, syn in results.items():
    print("=" * 60)
    annotation = annotate_anomaly(syn, anomaly_id, annotator)
    annotations[anomaly_id] = annotation
    show_annotation_result(
        syn.generated.image,
        annotation,
        title=f"Annotations — {anomaly_id} ({syn.method})",
    )
    print(save_annotation_artifact(f"{sample.name}_{anomaly_id}", annotation, output_dir / "nb1"))


---
## 4. Judge + retry (API VLM)

Unload diffusion / detector first (GPU VRAM). Judge uses **API vision chat** by default (not Qwen-Image-Edit). On `retry`, re-edit with Klein + a new seed up to `MAX_RETRIES`.


In [ ]:
import gc

import torch
from edgecase_synthesis.judge import VLMJudge, summarize_annotations
from edgecase_synthesis.viz import save_judge_artifact, show_judge_result


def _unload(*names: str) -> None:
    for name in names:
        obj = globals().get(name)
        if obj is not None and hasattr(obj, "unload"):
            obj.unload()
        globals()[name] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Free diffusion / detector VRAM before loading the VLM.
_unload("comparer", "annotator", "depth_model", "segmenter")

judge = VLMJudge.from_config(cfg)
source_hint = str(cfg.dataset.get("source_hint", "a real photograph"))
print(f"Judge backend={judge.backend}  model={judge.model_id}  threshold={judge.threshold}")
print("(accept if overall≥threshold and edge_case_present)")

judgments = {}
final_results = dict(results)

for anomaly_id, syn in list(results.items()):
    anomaly_cfg = load_anomaly(dataset, anomaly_id, start=PROJECT_ROOT)
    attempt = 0
    current = syn
    while True:
        result = judge.judge(
            current.generated.image,
            prompt=current.generated.prompt,
            anomaly_id=anomaly_id,
            anomaly_name=str(anomaly_cfg.get("display_name", anomaly_id)),
            annotations_summary=summarize_annotations(annotations.get(anomaly_id)),
            source_hint=source_hint,
        )
        print(f"{anomaly_id} attempt={attempt} → {result.decision} ({result.overall:.1f})")
        show_judge_result(
            current.generated.image,
            result,
            title=f"Judge — {anomaly_id} [{current.method}] attempt {attempt}",
        )
        print(result.rationale)
        print(save_judge_artifact(f"{sample.name}_{anomaly_id}_a{attempt}", result, output_dir / "nb1"))

        if result.decision != "retry" or attempt >= MAX_RETRIES:
            judgments[anomaly_id] = result
            final_results[anomaly_id] = current
            break

        attempt += 1
        print(f"  retrying {anomaly_id} with seed_offset={attempt} …")
        _unload("judge")
        depth_model = DepthEstimator.from_config(cfg)
        segmenter = Segmenter.from_config(cfg)
        comparer = MethodComparer.from_config(cfg)
        annotator = OpenVocabAnnotator.from_config(cfg)
        depth = depth_model.predict(sample.image)
        seg = segmenter.predict(sample.image)
        current = synthesize_one(
            sample.image,
            anomaly_id=anomaly_id,
            method=method_map[anomaly_id],
            cfg=cfg,
            comparer=comparer,
            depth=depth,
            segmentation=seg,
            project_root=PROJECT_ROOT,
            seed_offset=attempt,
        )
        # Re-annotate the new edit so the next judge call sees fresh labels.
        annotations[anomaly_id] = annotate_anomaly(current, anomaly_id, annotator)
        results[anomaly_id] = current
        _unload("comparer", "annotator", "depth_model", "segmenter")
        judge = VLMJudge.from_config(cfg)

accepted = sum(1 for r in judgments.values() if r.decision == "accept")
print(f"Acceptance: {accepted}/{len(judgments)}")
for aid, r in judgments.items():
    print(f"  {aid:16s} {r.decision:7s}  method={method_map[aid]}")


---
## Wrap-up

| Step | Notebook | Role |
|------|----------|------|
| Compare 5 edit paths | **1.5** | Pick generator; understand Klein vs Qwen vs API *edit* |
| Thin loop | **1** | Klein generate + API judge + retry |
| Batch + train | **2 → 3** | Same generator/judge at scale |

**Defaults:** `instruct` (Klein) + `judge.backend=api`.

Add an anomaly: YAML under `configs/datasets/<dataset>/generation/anomalies/` + set `METHOD_BY_ANOMALY`.
